# Padronizando colunas e corrigindo tipos

## Task 2: padronizar os nomes das colunas e corrigir os tipos

## O problema

- O público chega como texto, com vírgula de milhar (`"62,576"`), e a data como texto num formato próprio
  (`"13 Jun 1950 - 12:00"`).
- As colunas têm espaços (`Home Team Name`).

Objetivo: nomes simples, em português, e cada coluna com o tipo certo.

In [ ]:
import sys
sys.path.insert(0, "../../src")

from pyspark.sql import SparkSession, functions as F

spark = (
    SparkSession.builder
    .appName("arena-padronizacao-partidas")
    .master("local[*]")
    .config("spark.sql.warehouse.dir", "../../data/warehouse")
    .getOrCreate()
)

RAW = "../../data/raw"

spark

Começamos de onde a task 1 parou: o código do fim do vídeo 2.1.

## select: escolher e renomear colunas

`select` monta um DataFrame só com as colunas pedidas; `.alias()` dá o nome novo.

`F.col("Year")` aponta para uma coluna e aceita nomes com espaço.

## withColumn: corrigindo o público

`withColumn("nome", expressão)` cria uma coluna, ou substitui uma que já existe. Para mudar o tipo, `cast`:

`regexp_replace` troca um texto por outro dentro da coluna:

## withColumn: corrigindo a data

Formato `"13 Jun 1950 - 12:00"` = `"dd MMM yyyy - HH:mm"`. Converte com `to_timestamp`:

## Duas colunas novas

`total_gols` e `resultado` (`F.when(...).otherwise(...)`, o `CASE WHEN` do SQL). O resultado fica em `partidas`.

## Conferindo os vazios

Quantas partidas ficaram sem público ou sem data? A mesma contagem do vídeo 2.1: `filter` com `isNull()` e `count()`.

## Para consultar depois

- **Três jeitos de se referir a uma coluna:** `"ano"`, `F.col("ano")` e `jogos.ano`. Os três funcionam; o `F.col` é o
  mais robusto.
- **`try_cast` e `try_to_timestamp`:** fazem a mesma conversão, mas devolvem `null` em vez de erro quando o valor é
  inválido. Se usar, conte os nulos depois, para o problema não passar em silêncio.
- **`filter` com várias condições:** `&` (e), `|` (ou) e `~` (não), com cada condição entre parênteses, por exemplo
  `(F.col("ano") >= 1990) & (F.col("total_gols") > 5)`.

## Encerrando

Ao terminar o notebook, encerre a sessão Spark. Isso libera a memória e a porta da Spark UI para o próximo notebook.

In [ ]:
spark.stop()